# PostgreSQL Writer

This notebook consumes fraud predictions from Kafka and stores them into PostgreSQL.

Pipeline

Kafka
↓

fraud-predictions

↓

Spark Structured Streaming

↓

PostgreSQL

## Imports

In [ ]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    from_json
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

import psycopg2
import pandas as pd

## PostgreSQL Configuration

In [ ]:
POSTGRES_CONFIG = {

    "host": "localhost",

    "port": 5432,

    "database": "fraud_detection",

    "user": "postgres",

    "password": "Az@7306423517"

}

## Test PostgreSQL Connection

In [ ]:
try:

    conn = psycopg2.connect(**POSTGRES_CONFIG)

    print("✅ Connected to PostgreSQL")

    conn.close()

except Exception as e:

    print(e)

## Create Spark Session

In [ ]:
spark = (

    SparkSession.builder

    .appName("PostgreSQLWriter")

    .config(
        "spark.jars.packages",

        ",".join([

            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0",

            "org.postgresql:postgresql:42.7.4"

        ])

    )

    .getOrCreate()

)

spark.sparkContext.setLogLevel("ERROR")

## Stop Previous Streams

In [ ]:
for stream in spark.streams.active:

    stream.stop()

print("Previous streams stopped.")

## Read from Kafka

In [ ]:
kafka_df = (

    spark.readStream

    .format("kafka")

    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    )

    .option(
        "subscribe",
        "fraud-predictions"
    )

    .option(
        "startingOffsets",
        "latest"
    )

    .load()

)

## Define the Prediction Schema

In [ ]:
prediction_schema = StructType([

    StructField("transaction_id", StringType()),

    StructField("event_timestamp", StringType()),

    StructField("Amount", DoubleType()),

    StructField("Prediction", IntegerType()),

    StructField("Fraud_Probability", DoubleType()),

    StructField("Status", StringType()),

    StructField("Risk_Level", StringType())

])

## Parse the JSON

In [ ]:
prediction_df = (

    kafka_df

    .selectExpr("CAST(value AS STRING)")

    .select(

        from_json(

            col("value"),

            prediction_schema

        ).alias("data")

    )

    .select("data.*")

)

## Verify Schema

In [ ]:
prediction_df.printSchema()

## Preview Stream

In [ ]:
preview_query = (

    prediction_df

    .writeStream

    .format("console")

    .outputMode("append")

    .start()

)

# Write Predictions to PostgreSQL

# Write Batch to PostgreSQL

In [ ]:
def write_to_postgres(batch_df, batch_id):

    print("\n" + "=" * 70)
    print(f"Writing Batch {batch_id} to PostgreSQL")
    print("=" * 70)

    pdf = batch_df.toPandas()

    if pdf.empty:
        print("No records received.")
        return

    conn = None
    cursor = None

    try:

        conn = psycopg2.connect(**POSTGRES_CONFIG)

        cursor = conn.cursor()

        inserted = 0

        for _, row in pdf.iterrows():

            cursor.execute(
                """
                INSERT INTO fraud_predictions
                (
                    transaction_id,
                    event_timestamp,
                    amount,
                    prediction,
                    fraud_probability,
                    risk_level,
                    status
                )

                VALUES
                (%s,%s,%s,%s,%s,%s,%s)

                ON CONFLICT (transaction_id)
                DO NOTHING;
                """,

                (
                    row["transaction_id"],
                    row["event_timestamp"],
                    float(row["Amount"]),
                    int(row["Prediction"]),
                    float(row["Fraud_Probability"]),
                    row["Risk_Level"],
                    row["Status"]
                )

            )

            inserted += cursor.rowcount

        conn.commit()

        print(f"Inserted {inserted} row(s).")

    except Exception as e:

        print("Database Error")

        print(e)

        if conn:
            conn.rollback()

    finally:

        if cursor:
            cursor.close()

        if conn:
            conn.close()

# Start PostgreSQL Streaming Writer

In [ ]:
query = (

    prediction_df

    .writeStream

    .foreachBatch(write_to_postgres)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/postgres_writer"
    )

    .start()

)

## Streaming Status

In [ ]:
print(query.status)

## Active Streams

In [ ]:
spark.streams.active

## Keep the Stream Running

In [ ]:
query.awaitTermination()